# Demo 3: Advanced Analysis of Sleep and Activity Data

In this demo, we'll continue working with the Multilevel Monitoring of Activity and Sleep dataset to perform advanced analysis of dense physiological signals. We'll learn how to:
- Process and analyze dense accelerometer data
- Apply signal processing techniques to remove noise
- Extract meaningful features from raw sensor data
- Connect activity patterns to health outcomes

## 1. Setting Up and Loading the Data

First, let's import the necessary libraries and download the dataset from PhysioNet.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft, fftfreq
import os
from urllib.request import urlretrieve

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Create a directory for data if it doesn't exist
os.makedirs('data', exist_ok=True)

In [ ]:
# Download data from PhysioNet
base_url = "https://physionet.org/files/mmash/1.0.0/"

# Download info.csv file if not already downloaded
info_url = base_url + "info.csv"
info_file = "data/mmash_info.csv"
if not os.path.exists(info_file):
    urlretrieve(info_url, info_file)

# Download accelerometer data for one subject
subject_id = '1'
accel_url = base_url + f"user_{subject_id}/Actigraph.csv"
accel_file = f"data/actigraph_{subject_id}.csv"
urlretrieve(accel_url, accel_file)

# Download sleep data for the same subject
sleep_url = base_url + f"user_{subject_id}/sleep.csv"
sleep_file = f"data/sleep_{subject_id}.csv"
if not os.path.exists(sleep_file):
    urlretrieve(sleep_url, sleep_file)

print(f"Downloaded accelerometer and sleep data for subject {subject_id}")

Now let's load the accelerometer data and examine it.

In [ ]:
# Load accelerometer data
accel_data = pd.read_csv(accel_file)

# Display the first few rows
print("Accelerometer data:")
accel_data.head()

In [ ]:
# Basic statistics
print(f"Data points: {len(accel_data)}")
print(f"Columns: {accel_data.columns.tolist()}")

# Check for missing values
print("\nMissing values:")
print(accel_data.isnull().sum())

# Convert timestamp to datetime if needed
if 'timestamp' in accel_data.columns:
    accel_data['timestamp'] = pd.to_datetime(accel_data['timestamp'])
    
    # Calculate sampling rate
    time_diff = accel_data['timestamp'].diff().median()
    sampling_rate = 1 / time_diff.total_seconds()
    print(f"\nEstimated sampling rate: {sampling_rate:.2f} Hz")
    print(f"Time range: {accel_data['timestamp'].min()} to {accel_data['timestamp'].max()}")

## 2. Visualizing Raw Accelerometer Data

Let's visualize the raw accelerometer data to understand its characteristics.

In [ ]:
# Plot a segment of the accelerometer data
# Let's take a 10-minute segment (adjust based on actual data)
if 'timestamp' in accel_data.columns:
    start_time = accel_data['timestamp'].min()
    end_time = start_time + pd.Timedelta(minutes=10)
    segment = accel_data[(accel_data['timestamp'] >= start_time) & 
                         (accel_data['timestamp'] <= end_time)]
else:
    # If no timestamp, just take the first 1000 rows
    segment = accel_data.iloc[:1000]

# Plot the three axes
plt.figure(figsize=(12, 8))

# X-axis
plt.subplot(3, 1, 1)
if 'timestamp' in segment.columns:
    plt.plot(segment['timestamp'], segment['x'], 'r-')
else:
    plt.plot(segment.index, segment['x'], 'r-')
plt.title('X-axis Acceleration')
plt.ylabel('Acceleration')
plt.grid(True)

# Y-axis
plt.subplot(3, 1, 2)
if 'timestamp' in segment.columns:
    plt.plot(segment['timestamp'], segment['y'], 'g-')
else:
    plt.plot(segment.index, segment['y'], 'g-')
plt.title('Y-axis Acceleration')
plt.ylabel('Acceleration')
plt.grid(True)

# Z-axis
plt.subplot(3, 1, 3)
if 'timestamp' in segment.columns:
    plt.plot(segment['timestamp'], segment['z'], 'b-')
else:
    plt.plot(segment.index, segment['z'], 'b-')
plt.title('Z-axis Acceleration')
plt.xlabel('Time')
plt.ylabel('Acceleration')
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Signal Processing Techniques

Let's apply some signal processing techniques to clean and analyze the accelerometer data.

In [ ]:
# Calculate the magnitude of acceleration (vector magnitude)
accel_data['magnitude'] = np.sqrt(accel_data['x']**2 + accel_data['y']**2 + accel_data['z']**2)

# Plot the magnitude
plt.figure(figsize=(12, 6))
if 'timestamp' in accel_data.columns:
    plt.plot(segment['timestamp'], segment['magnitude'], 'k-')
else:
    plt.plot(segment.index, segment['magnitude'], 'k-')
plt.title('Acceleration Magnitude')
plt.xlabel('Time')
plt.ylabel('Magnitude')
plt.grid(True)
plt.show()

In [ ]:
# Apply a low-pass filter to remove high-frequency noise
def apply_lowpass_filter(data, column, cutoff, fs, order=5):
    """Apply a low-pass filter to the data.
    
    Parameters:
    - data: DataFrame containing the data
    - column: Column name to filter
    - cutoff: Cutoff frequency in Hz
    - fs: Sampling frequency in Hz
    - order: Filter order
    
    Returns:
    - Filtered data
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = signal.filtfilt(b, a, data[column])
    return filtered_data

# Estimate sampling frequency if not already calculated
if 'timestamp' in accel_data.columns and 'sampling_rate' not in locals():
    time_diff = accel_data['timestamp'].diff().median()
    sampling_rate = 1 / time_diff.total_seconds()
else:
    # Assume a common sampling rate for accelerometers if not available
    sampling_rate = 30  # Hz

# Apply low-pass filter to magnitude
cutoff_freq = 1.0  # Hz - adjust based on your needs
accel_data['magnitude_filtered'] = apply_lowpass_filter(
    accel_data, 'magnitude', cutoff_freq, sampling_rate)

# Plot original vs. filtered magnitude
plt.figure(figsize=(12, 6))
if 'timestamp' in segment.columns:
    plt.plot(segment['timestamp'], segment['magnitude'], 'k-', alpha=0.3, label='Original')
    plt.plot(segment['timestamp'], segment['magnitude_filtered'].iloc[:len(segment)], 
             'r-', linewidth=2, label='Filtered')
else:
    plt.plot(segment.index, segment['magnitude'], 'k-', alpha=0.3, label='Original')
    plt.plot(segment.index, segment['magnitude_filtered'].iloc[:len(segment)], 
             'r-', linewidth=2, label='Filtered')
plt.title(f'Original vs. Low-Pass Filtered (Cutoff: {cutoff_freq} Hz)')
plt.xlabel('Time')
plt.ylabel('Magnitude')
plt.legend()
plt.grid(True)
plt.show()

## 4. Feature Extraction from Accelerometer Data

Let's extract meaningful features from the accelerometer data that can be used for activity recognition and health monitoring.

In [ ]:
# Function to extract features from a window of accelerometer data
def extract_features(data, window_size=100):
    """Extract features from accelerometer data using a sliding window approach.
    
    Parameters:
    - data: DataFrame containing accelerometer data
    - window_size: Number of samples in each window
    
    Returns:
    - DataFrame with extracted features
    """
    features = []
    
    # Ensure we have timestamp information
    if 'timestamp' in data.columns:
        time_col = 'timestamp'
    else:
        # Create a dummy timestamp column
        data = data.copy()
        data['timestamp'] = pd.date_range(
            start='2023-01-01', periods=len(data), freq=f'{1/sampling_rate:.6f}S')
        time_col = 'timestamp'
    
    # Process data in windows with 50% overlap
    step_size = window_size // 2
    for i in range(0, len(data) - window_size, step_size):
        window = data.iloc[i:i+window_size]
        
        # Time domain features
        feature_dict = {
            'start_time': window[time_col].iloc[0],
            'end_time': window[time_col].iloc[-1],
            
            # Mean
            'mean_x': window['x'].mean(),
            'mean_y': window['y'].mean(),
            'mean_z': window['z'].mean(),
            'mean_mag': window['magnitude'].mean(),
            
            # Standard deviation
            'std_x': window['x'].std(),
            'std_y': window['y'].std(),
            'std_z': window['z'].std(),
            'std_mag': window['magnitude'].std(),
            
            # Min/Max
            'min_mag': window['magnitude'].min(),
            'max_mag': window['magnitude'].max(),
            'range_mag': window['magnitude'].max() - window['magnitude'].min(),
        }
        
        features.append(feature_dict)
    
    return pd.DataFrame(features)

In [ ]:
# Extract features from the accelerometer data
# Use a window size of approximately 5 seconds
window_size = int(5 * sampling_rate)
features_df = extract_features(accel_data, window_size=window_size)

# Display the extracted features
print(f"Extracted {len(features_df)} feature windows from {len(accel_data)} data points")
print(f"Features: {features_df.columns.tolist()}")
features_df.head()

In [ ]:
# Visualize some of the extracted features over time
plt.figure(figsize=(12, 8))

# Mean magnitude
plt.subplot(2, 1, 1)
plt.plot(features_df['start_time'], features_df['mean_mag'], 'b-')
plt.title('Mean Acceleration Magnitude')
plt.ylabel('Mean Magnitude')
plt.grid(True)

# Standard deviation of magnitude
plt.subplot(2, 1, 2)
plt.plot(features_df['start_time'], features_df['std_mag'], 'r-')
plt.title('Standard Deviation of Acceleration Magnitude')
plt.xlabel('Time')
plt.ylabel('Std Magnitude')
plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Activity Recognition

Let's use the extracted features to identify different activity patterns.

In [ ]:
# Simple activity recognition based on thresholds
def classify_activity(features):
    """Classify activity based on simple thresholds.
    
    This is a simplified approach. In practice, you would use machine learning.
    
    Parameters:
    - features: DataFrame with extracted features
    
    Returns:
    - DataFrame with activity labels
    """
    features = features.copy()
    
    # Define thresholds (these would be determined empirically)
    # These are just examples and should be adjusted based on your data
    sedentary_threshold = 0.1
    light_threshold = 0.3
    moderate_threshold = 0.7
    
    # Classify based on mean magnitude and standard deviation
    conditions = [
        (features['mean_mag'] < sedentary_threshold),
        (features['mean_mag'] < light_threshold) & (features['std_mag'] < light_threshold),
        (features['mean_mag'] < moderate_threshold) | (features['std_mag'] < moderate_threshold),
        (features['mean_mag'] >= moderate_threshold) & (features['std_mag'] >= moderate_threshold)
    ]
    choices = ['Sedentary', 'Light Activity', 'Moderate Activity', 'Vigorous Activity']
    
    features['activity'] = np.select(conditions, choices, default='Unknown')
    
    return features

In [ ]:
# Classify activities
activity_df = classify_activity(features_df)

# Display activity distribution
activity_counts = activity_df['activity'].value_counts()
print("Activity distribution:")
print(activity_counts)

# Plot activity distribution
plt.figure(figsize=(10, 6))
activity_counts.plot(kind='bar', color='skyblue')
plt.title('Activity Distribution')
plt.xlabel('Activity Type')
plt.ylabel('Count')
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize activities over time
plt.figure(figsize=(12, 6))

# Create a numeric mapping for activities for visualization
activity_map = {
    'Sedentary': 0,
    'Light Activity': 1,
    'Moderate Activity': 2,
    'Vigorous Activity': 3,
    'Unknown': -1
}
activity_df['activity_num'] = activity_df['activity'].map(activity_map)

# Plot activities
plt.scatter(activity_df['start_time'], activity_df['activity_num'], 
            c=activity_df['activity_num'], cmap='viridis', 
            alpha=0.7, s=30)

# Add mean magnitude as a line for reference
ax1 = plt.gca()
ax2 = ax1.twinx()
ax2.plot(activity_df['start_time'], activity_df['mean_mag'], 'r-', alpha=0.5)
ax2.set_ylabel('Mean Magnitude', color='r')

# Customize the plot
plt.title('Activity Classification Over Time')
plt.xlabel('Time')
ax1.set_ylabel('Activity Type')
ax1.set_yticks(list(activity_map.values()))
ax1.set_yticklabels(list(activity_map.keys()))
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Summary and Discussion

In this demo, we've explored advanced analysis of dense accelerometer data. Here's what we've learned:

1. **Signal Processing**: We applied filtering techniques to remove noise from accelerometer data, making patterns more visible.

2. **Feature Extraction**: We extracted meaningful features from raw accelerometer data using a sliding window approach, including statistical measures like mean, standard deviation, and range.

3. **Activity Recognition**: We implemented a simple threshold-based activity recognition system to classify activities as sedentary, light, moderate, or vigorous.

4. **Visualization**: We created visualizations to understand activity patterns over time.

### Key Insights:

- Dense physiological data requires preprocessing to extract meaningful information.
- Simple features like mean and standard deviation can be surprisingly effective for activity recognition.
- Visualizing activity patterns over time provides insights into daily behavior.

### Next Steps:

- Apply machine learning for more accurate activity recognition.
- Connect activity patterns to health outcomes like sleep quality.
- Explore frequency domain features for more detailed analysis.
- Implement real-time processing for continuous monitoring applications.